# Reasoning, RAG and Multi-Agent Setup (ReAct Agents)

In [1]:
import os
import yfinance as yf
from dotenv import load_dotenv
from ddgs import DDGS
from openai import OpenAI
import lancedb
from lancedb.embeddings import get_registry
import PyPDF2
import pandas as pd

In [3]:
# 1. Loading environment variables
try:
    load_dotenv()
except Exception as e:
    print(f"Error loading env variables: {e}")

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


# 2. Downloading and extracting PDF text from path
def extract_text_from_pdf(file="nvidia_report.pdf"):
    try:
        print(" Downloading PDF...")
        reader = PyPDF2.PdfReader(file)
        file_content = ''
        for page in reader.pages:
            file_content += " " + page.extract_text()

        return file_content.strip()
    except Exception as e:
        print(f"Erro download or extrating text from PDF: {e}")
        return "" 


# 3. Creating Vetorial Database
db_uri = "rag/lancedb"
os.makedirs(db_uri, exist_ok=True)

try:
    print("Starting vectorial database LanceDB...")
    db = lancedb.connect(db_uri)

    if "vectordb" in db.list_tables():
        table = db.open_table("vectordb")
        print("Existent vectorial database loaded.")
    else:
        # Download and process the PDF
        # url_pdf = "https://s201.q4cdn.com/141608511/files/doc_financials/2024/ar/NVIDIA-2024-Annual-Report.pdf"
        pdf_text = extract_text_from_pdf()

        # pdf_text already extracted and generating blocks:
        blocks = [pdf_text[i:i+2000] for i in range(0, len(pdf_text), 2000)]

        print("Generating embeddings using the OpenAI api...")

        # 1) Call embeddings
        try:
            emb_response = client.embeddings.create(
                model="text-embedding-3-small",
                input=blocks  # lista de strings
            )
            # 2) Extracting embeddings
            embeddings = [item.embedding for item in emb_response.data]

            # 3) Creating the dataframe
            data = pd.DataFrame({
                "id": range(len(blocks)),
                "text": blocks,
                "embedding": embeddings
            })

            # 4) Creating a table on LanceDB (Lance will allow 'embedding' column as vector)
            table = db.create_table("vectordb", data=data)
            print("Vector database successfuly created.")

        except Exception as e:
            print("Erro generating OpenAI embeddings:", e)
except Exception as e:
    print(f"Erro creating or loading vector database: {e}")

Starting vectorial database LanceDB...
Generating embeddings using the OpenAI api...
Vector database successfuly created.


In [4]:
# 4. Agents
class SearchAgent:
    def search(self, query):
        try:
            with DDGS() as ddgs:
                results = list(ddgs.text(query, max_results=3))
            return results
        except Exception as e:
            return [f"Search error: {e}"]


class FinancialAgent:
    def __init__(self, ticker="NVDA"):
        self.ticker = ticker

    def collect_data(self):
        try:
            adction = yf.Ticker(self.ticker)
            data = {
                "company": adction.info.get("longName"),
                "price": adction.history(period="1d")["Close"].iloc[-1],
                "sector": adction.info.get("sector"),
            }
            return data
        except Exception as e:
            return {"error": str(e)}

In [5]:
# 5. RAG function (vectorial search)
def rag_search(query, table, client, top_k=3):
    try:
        print("Generating embedding from query and searching on RAG...")

        emb_response = client.embeddings.create(model="text-embedding-3-small", input=query)

        embedding_query = emb_response.data[0].embedding

        # Vectorial search on RAG unsing LanceDB
        results = (table.search(embedding_query).limit(top_k).to_pandas())

        print(f"✅ {len(results)} results found on RAG.")
        return results["text"].tolist()

    except Exception as e:
        print(f"Erro searching on RAG: {e}")
        return [f"Erro searching on RAG: {e}"]

In [6]:
def reasoning_process(question, search_results, finance_data, rag_context):
    steps = [
        "1️⃣ Analyze the legal and regulatory context found in the web search.",
        "2️⃣ Examine Nvidia's financial and market information.",
        "3️⃣ Integrate information from the annual report (RAG).",
        "4️⃣ Generate a conclusion with logical explanation in natural language."
    ]

    full_prompt = f"""
        You are a financial and legal analyst.
        Carefully follow the reasoning process below (Chain of Thought):

        Steps:
        {chr(10).join(steps)}

        ---
        🔍 Search results:
        {search_results}

        💹 Financial data:
        {finance_data}

        📚 Context extracted from the annual report (RAG):
        {rag_context}

        ---
        Question: {question}

        Answer in English, explaining your reasoning in a clear and structured way.
    """

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": full_prompt}],
        temperature=0.6
    )

    return response.choices[0].message.content.strip()

In [7]:
if __name__ == "__main__":
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    db = lancedb.connect("rag/lancedb")
    table = db.open_table("vectordb")
    question = "Is there any relation between legal aspects and the financial data of Nvidia?"

    web_agent = SearchAgent()
    fin_agent = FinancialAgent()

    search_results = web_agent.search("Legal Aspects of Nvidia 2024") # It could call a LLM to format the output data.
    financial_data = fin_agent.collect_data()
    rag_context = rag_search("Legal and financial regulation of Nvidia 2024", table, client)  # It could be a dynamic system
    rag_context = " - ".join(rag_context)

    print("\n🧠 Starting reasoning process...\n")
    finan_answer = reasoning_process(question, search_results, financial_data, rag_context)

    print("✅ Final Answer:\n")
    print(finan_answer)

Generating embedding from query and searching on RAG...
✅ 3 results found on RAG.

🧠 Starting reasoning process...

✅ Final Answer:

To determine if there is any relation between the legal aspects and the financial data of Nvidia, let's follow the outlined steps:

1️⃣ **Analyze the legal and regulatory context:**
   - Nvidia is facing a complex legal environment, which includes scrutiny from regulators in the EU, UK, and China regarding their AI chip sales and investment practices. Specifically, China's State Administration for Market Regulation (SAMR) has accused Nvidia of antitrust violations related to its 2016 acquisition of Mellanox.
   - Nvidia has a significant number of legal filings and decisions, including federal litigation and state court decisions, indicating an active legal landscape.
   - There are ongoing legal challenges, such as the derivative action in Delaware, alleging breach of fiduciary duty and insider trading linked to cryptocurrency mining impacts on GPU deman

In [8]:
for res in search_results:
    print(res)

{'title': 'PDF NVIDIA Opening Brief 8-13-24 Final - Supreme Court of the United States', 'href': 'https://www.supremecourt.gov/DocketPDF/23/23-970/322355/20240813143333491_NVIDIA+Opening+Brief+8-13-24+Final.pdf', 'body': 'On March 4, 2024, Petitioners filed a timely petition for certiorari, which the Court granted on June 17, 2024. This Court has jurisdiction under 28 U.S.C. § 1254(1).'}
{'title': 'Nvidia Profile - Nvidia Trademarks, Patents, Litigation Filings and ...', 'href': 'https://companyprofiles.justia.com/company/nvidia', 'body': 'Nvidia legal profile including 351 Trademarks, 5946 Patent Grants, 1848 Patent Applications, 303 Federal Litigation Filings, 292 Federal District Court Decisions, 25 State Court Decisions and 7 Federal Appellate Court Decisions'}
{'title': "Nvidia's Legal Risks and Competitive Positioning in AI and Autonomous ...", 'href': 'https://www.ainvest.com/news/nvidia-legal-risks-competitive-positioning-ai-autonomous-driving-navigating-trade-secrets-litigatio

In [9]:
print(financial_data)

{'company': 'NVIDIA Corporation', 'price': np.float64(218.2899932861328), 'sector': 'Technology'}


In [10]:
print(rag_context)

ct upon our capital expenditures, 
results of operations, or competitive position and we do not currently anticipate material capital expenditures for 
environmental control facilities. Compliance with existing or future governmental regulations, including, but not limited 
to, those pertaining to IP ownership and infringement, taxes, import and export requirements and tariffs, anti-corruption, 
business acquisitions, foreign exchange controls and cash repatriation restrictions, data privacy requirements, 
competition and antitrust, advertising, employment, product regulations, cybersecurity, environmental, health and safety 
requirements, the responsible use of AI, climate change, cryptocurrency, and consumer laws, could increase our costs, 
impact our competitive position, and otherwise may have a material adverse impact on our business, financial condition 
and results of operations in subsequent periods. Refer to “Item 1A. Risk Factors” for a discussion of these potential 
impacts.